In [2]:
import sys
sys.path.append('../../')

In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
import pandas as pd
import torch

from tokenizers import ByteLevelBPETokenizer

from src.config.paths import DATA_PATH

In [5]:
N_VOCAB = 10_000
CHUNK_SIZE = 32

# Tokenizers

In [6]:
# with open(DATA_PATH.joinpath('flattened_corpora.txt'), 'w+') as f:
#     for i in [IQ2_PATH, SUPREME_PATH, TENNIS_PATH]:
#         df = pd.read_json(i.joinpath('utterances.jsonl'), lines=True)
#         f.write('\n'.join(df['text']))

In [7]:
tok = ByteLevelBPETokenizer()

In [8]:
tok.train(
    files=[DATA_PATH.joinpath('flattened_corpora.txt').__str__()],
    vocab_size=N_VOCAB,
    min_frequency=2,
    special_tokens=['[SEP]']
)

In [9]:
tok.get_vocab().keys()

dict_keys(['selling', 'ĠIndones', 'Ġpunish', 'Ġexcellent', 'gery', 'ĠVan', 'Ġparticipation', 'Ġcommissioner', 'ification', 'acket', 'se', 'ĠSandy', 'Ġintellectual', 'ĠWadhwa', 'Ġzone', 'ĠBernstein', 'oss', 'Ġinfin', 'Act', 'ane', 'ĠCEO', 'eating', 'Ġmag', 'Ġfossil', 'Ġinvade', 'Ġdimension', 'Ġunacceptable', 'Ġfemale', 'Ġs', 'ico', 'ğ', 'ermann', 'ĠDie', '500', 'Ġprem', 'Ġ34', 'Bl', 'Ġscen', 'Ġmethods', 'Ġreturns', 'amin', 'Ġdraft', 'Ġworry', 'ĠLincoln', 'iling', 'Ġdown', 'ĠFeb', 'Ġargue', 'Ġcaught', 'ians', 'ĠPear', 'Ġintense', 'Ġbans', 'ang', 'ĠPush', 'Ġstyle', 'ĠAl', 'alid', 'oun', 'mond', 'Ġwip', 'ĠHel', 'ĠBlair', 'Ġimprove', 'Ġlaunch', 'crib', 'Ġequipment', 'ĠWhere', 'Ġtweet', 'winning', 'Ġreceiving', 'Ġreached', 'oughly', 'Ġanalysis', 'ohol', 'Ġextended', 'Ġpen', 'iger', 'X', 'asure', 'ĠHubb', '>', 'Ġbed', 'Well', 'acare', 'Ġexcuse', 'Ġnat', 'ments', 'era', 'ried', 'ĠFer', 'areed', 'Ġseize', 'Ġarbitrary', 'Af', 'iving', 'Ă', 'Ġdisclosure', 'oner', 'Ġharms', 'Ġdepends', 'Ġbaby', 'Ġ

In [10]:
tok.encode('yoo test [SEP] test').tokens

['y', 'oo', 'Ġtest', 'Ġ', '[SEP]', 'Ġtest']

# Dataset

In [11]:
df = pd.read_parquet(DATA_PATH.joinpath('dia-merged2.parquet'))

## Conversation Chunking

In [12]:
counts = df.groupby('conversation_id')['id'].count()
counts 

conversation_id
10           160
100          101
1004          36
1007         167
1010         131
            ... 
iq2_897     1087
iq2_9204    1182
iq2_9437    1184
iq2_9739    1290
iq2_9973    1379
Name: id, Length: 3151, dtype: int64

In [13]:
counts.max(), counts.mean(), counts.mode()[0], counts.min()

(np.int64(1665), np.float64(187.71405902887972), np.int64(92), np.int64(4))

In [14]:
for limit in [16, 32, 64]:
    outliers = (counts < limit)
    print(limit, '-', outliers.sum(), '-', f'{outliers.sum() / len(counts):%}')

16 - 45 - 1.428118%
32 - 198 - 6.283719%
64 - 807 - 25.610917%


I am torn between 32 and 64 right now but leaning 32
- Only 6.28% of convos would be dropped if i did chunking with a size of 32, 
- its reasonably large to provide good context to the average sentence within a chunk
- its not so big it would slow down training

Obviously samples from the non-outlier convos would be dropped too but I guess I can go back to 16 if its too bad

I guess if even after chunking I feel I have too much data, I might opt for 64, but that can definitely get rough on training so probably gonna stick with 32

In [15]:
res = []

chunk_idx = -1
in_chunk_idx = 0
last_convo = None

for _, row in df.iterrows():
    if not last_convo or last_convo != row.conversation_id:
        last_convo = row.conversation_id
        chunk_idx += 1
        in_chunk_idx = 0

    res.append(chunk_idx)

    in_chunk_idx += 1 
    if CHUNK_SIZE == in_chunk_idx:
        in_chunk_idx = 0
        chunk_idx += 1

In [16]:
df['chunk_id'] = res

In [17]:
counts = df.groupby('chunk_id')['id'].count()
filled_chunks = counts[counts == CHUNK_SIZE].index.to_numpy()
df = df[df['chunk_id'].isin(filled_chunks)]

In [18]:
ids_oton = {v:k for k,v in enumerate(df['chunk_id'].unique())}
df['chunk_id'] = df['chunk_id'].apply(lambda x: ids_oton[x])

In [20]:
df['chunk_id'].unique()

array([    0,     1,     2, ..., 16924, 16925, 16926], shape=(16927,))

In [ ]:
df.groupby('root')['id'].count

root
iq2    127456
sup    206432
ten    207776
Name: id, dtype: int64